# Tutorial: Integrals and Density Matrices

---

This tutorial provides examples of how to use Forte2 to obtain integrals and reduced density matrices in the molecular orbital basis.
It can be used as a basis to implement custom post-HF and post-CASSCF methods that require these quantities, such as multireference perturbation theories and multireference coupled cluster theories.

## Integrals from restricted Hartree–Fock computations

We begin with a restricted open-shell Hartree–Fock case and first run a SCF computation on the doublet state of the HF<sup>+</sup> cation.

In [1]:
import numpy as np
from forte2 import RHF, System

xyz = """
H 0.000 0.000 0.000
F 0.000 0.000 1.000
"""

system = System(xyz=xyz, basis_set="cc-pVDZ", auxiliary_basis_set="cc-pVTZ-JKFIT")

rhf = RHF(charge=0)(system)
rhf.run()

[mods_manager] loading mod determinant_printing from /Users/fevange/.forte2/mods
Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   H   0.00000000   0.00000000   0.00000000
   F   0.00000000   0.00000000   1.88972612
Parsed 2 atoms with basis set of 19 functions.
  Max eigenvalue: 2.798e+00
  Min eigenvalue: 6.026e-02
  Condition number: 4.643e+01
  Inverse condition number: 2.154e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 19
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 6.026e-02
Number of electrons: 10
Number of alpha electrons: 5
Number of beta electrons: 5
Ms: 0
Total charge: 0
Number of basis functions: 19
Number of orthogonalized basis functions: 19
Number of auxiliary basis functions: 109
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to stori

RHF(charge=0, do_diis=True, diis_start=1, diis_nvec=8, diis_min=2, econv=1e-09, dconv=1e-06, maxiter=100, guess_type='minao', level_shift=None, level_shift_thresh=1e-05, die_if_not_converged=True, executed=True, converged=True)

Forte2 provides the helper class `RestrictedMOIntegrals` to compute integrals in the restricted case. This class requires the user to specify:
- A `System` object.
- A matrix of MO coefficients (`C`).
- A list of orbitals (`orbitals`) for which we want to compute the molecular integrals.
- An optional list of doubly occupied core orbitals (`core_orbitals`).
 
`RestrictedMOIntegrals` computes the following quantities defined over **spatial orbital indices**:
- **Core energy**. This is a scalar term that contains the nuclear-nuclear repulsion energy ($V_\mathrm{NN}$) plus the one- and two-electron contributions from the core orbitals:
\begin{equation}
E_\mathrm{core} = V_\mathrm{NN} + 2 \sum_{i}^{\mathrm{core}} h_{ii} + \sum_{ij}^{\mathrm{core}} [2 \langle ij | ij\rangle - \langle ij | ji\rangle].
\end{equation}
- **One-electron integrals**. These are the bare one-electron integrals ($h_{pq}$, kinetic + potential energy) plus two-electron contributions from the core orbitals:
\begin{equation}
h'_{pq} = h_{pq} + \sum_{i}^{\mathrm{core}} [2 \langle pi | qi\rangle - \langle pi | iq\rangle], \quad p,q \in \mathrm{orbitals}.
\end{equation}
- **Two-electron integrals**. This is a scalar term that contains the nuclear-nuclear repulsion energy ($V_\mathrm{NN}$) plus the one- and two-electron contributions from the core orbitals:
\begin{equation}
v_{pqrs} = \langle pq | rs\rangle, \quad p,q,r,s \in \mathrm{orbitals}.
\end{equation}

Below we compute all the integral and freeze the lowest HF orbital, which corresponds to the F 1s core orbital. To pass the orbital coefficients, we access the attribute `C` of the `RHF` object. This quantity is an array with dimensions dependent on the type of HF computation (e.g., restricted, unrestricted, general). For RHF, this list had dimension 1 and the coefficient matrix is given by `rhf.C[0]`:

In [2]:
from forte2.jkbuilder.mointegrals import RestrictedMOIntegrals

# total number of orbitals
norb = rhf.nmo

# freeze 1s orbital of fluorine
core_orbitals = [0]

# orbitals are all orbitals except the core orbitals
orbitals = sorted(list(set(range(norb)) - set(core_orbitals)))

mo_integrals = RestrictedMOIntegrals(system=system, C=rhf.C[0],orbitals=orbitals, core_orbitals=core_orbitals)

Below we print the value of $E_\mathrm{core}$:

In [3]:
print(mo_integrals.E)

-71.77315462576419


We can now verify that the integrals are correct by computing the Hartree–Fock energy using Slater rules:
\begin{equation}
E_\mathrm{HF} = E_\mathrm{core} + 2 \sum_{i}^{\mathrm{occ}} h_{ii} + \sum_{ij}^{\mathrm{occ}} [2 \langle ij | ij\rangle - \langle ij | ji\rangle].
\end{equation}

In [4]:
# number of occupied orbitals minus the core orbitals
nocc = rhf.na - len(core_orbitals)

# extract the occupied blocks of the one- and two-electron integrals
Hocc = mo_integrals.H[:nocc, :nocc]
Vocc = mo_integrals.V[:nocc, :nocc, :nocc, :nocc] 

# compute the Hartree-Fock energy using the integrals
Ehf = mo_integrals.E + 2 * np.einsum("ii->",Hocc) + 2 *np.einsum("ijij->", Vocc) - np.einsum("ijji->", Vocc)

print(f"RHF energy from SCF:       {rhf.E:.12f} Eh")
print(f"RHF energy from integrals: {Ehf:.12f} Eh")

RHF energy from SCF:       -100.009873562527 Eh
RHF energy from integrals: -100.009873562527 Eh


## Integrals in the spinorbital basis

When testing new methods it is often convenient to work directly in the spinorbital basis. Forte2 provides the helper class `SpinorbitalIntegrals` to evaluate integrals in the spinorbital basis.

`SpinorbitalIntegrals` computes the following quantities defined over **spinorbital indices**:
- **Core energy**. This is a scalar term that contains the nuclear-nuclear repulsion energy ($V_\mathrm{NN}$) plus the one- and two-electron contributions from the core spinorbitals:
\begin{equation}
E_\mathrm{core} = V_\mathrm{NN} + \sum_{i}^{\mathrm{core}} h_{ii} + \frac{1}{2} \sum_{ij}^{\mathrm{core}} [\langle ij | ij\rangle - \langle ij | ji\rangle].
\end{equation}
- **One-electron integrals**. These are the bare one-electron integrals ($h_{pq}$, kinetic + potential energy) plus two-electron contributions from the core spinorbitals:
\begin{equation}
h'_{pq} = h_{pq} + \sum_{i}^{\mathrm{core}} [\langle pi | qi\rangle - \langle pi | iq\rangle], \quad p,q \in \mathrm{spinorbitals}.
\end{equation}
- **Two-electron integrals**. This is a scalar term that contains the nuclear-nuclear repulsion energy ($V_\mathrm{NN}$) plus the one- and two-electron contributions from the core spinorbitals:
\begin{equation}
v_{pqrs} = \langle pq | rs\rangle, \quad p,q,r,s \in \mathrm{spinorbitals}.
\end{equation}

Below we compute all the integral and freeze the lowest two orbitals, which correspond to the F 1s core spinorbitals.

To use this class **we need to promote the orbital coefficients to the spinorbital basis**. This can be done via the function `convert_coeff_spatial_to_spinor`. Note that **the final spinorbital basis will be ordered such that all α and β spinorbitals alternate** as in:
\begin{equation}
\{\psi^{\alpha}_{1}, \psi^{\beta}_{1}, \psi^{\alpha}_{2}, \psi^{\beta}_{2}, \psi^{\alpha}_{3}, \psi^{\beta}_{3}, \cdots \}.
\end{equation}

The `System` object also needs to be aware that we are working in the spinorbital basis, so we set `system.two_component = True`:

In [5]:
from forte2.jkbuilder.mointegrals import SpinorbitalIntegrals
from forte2.scf.scf_utils import convert_coeff_spatial_to_spinor

# convert RHF to a two-component orbital
Cso = convert_coeff_spatial_to_spinor(rhf.C)[0]

# total number of spinorbitals
nso = rhf.nmo * 2

# freeze 1s orbital of fluorine
core_spinorbitals = [0, 1]

# orbitals are all orbitals except the core orbitals
orbitals = sorted(list(set(range(nso)) - set(core_spinorbitals)))

system.two_component = True

so_integrals = SpinorbitalIntegrals(system=system, C=Cso,spinorbitals=orbitals, core_spinorbitals=core_spinorbitals)

Using the integrals in the spinorbital basis we can verify that the energy is the same as the RHF one but evaluated using the spinorbital expression:
\begin{equation}
E_\mathrm{HF} = E_\mathrm{core} + \sum_{i}^{\mathrm{occ}} h_{ii} + \frac{1}{2} \sum_{ij}^{\mathrm{occ}} [\langle ij | ij\rangle - \langle ij | ji\rangle].
\end{equation}

In [6]:
# number of occupied spinorbitals minus the core spinorbitals
noccso = rhf.na + rhf.nb - len(core_spinorbitals)

# extract the occupied blocks of the one- and two-electron integrals
Hoccso = so_integrals.H[:noccso, :noccso]
Voccso = so_integrals.V[:noccso, :noccso, :noccso, :noccso] 
    
# compute the Hartree-Fock energy using the integrals (spinorbital version of the formula)
Ehfso = so_integrals.E + np.einsum("ii->",Hoccso) + 0.5 * np.einsum("ijij->", Voccso) - 0.5 * np.einsum("ijji->", Voccso)

print(f"RHF energy from spinorbital integrals: {Ehfso}")

RHF energy from spinorbital integrals: (-100.00987356252674+0j)


## Integrals from generalized Hartree–Fock (GHF) computations

Next we consider the case of a generalize Hartree–Fock computation. Let's start from an input computation:

In [7]:
import numpy as np
from forte2 import GHF

xyz = """
H 0.000 0.000 0.000
F 0.000 0.000 1.000
"""

system = System(xyz=xyz, basis_set="cc-pVDZ", auxiliary_basis_set="cc-pVTZ-JKFIT")

ghf = GHF(charge=1)(system)
ghf.run()

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   H   0.00000000   0.00000000   0.00000000
   F   0.00000000   0.00000000   1.88972612
Parsed 2 atoms with basis set of 19 functions.
  Max eigenvalue: 2.798e+00
  Min eigenvalue: 6.026e-02
  Condition number: 4.643e+01
  Inverse condition number: 2.154e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 19
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 6.026e-02
Number of electrons: 9
Total charge: 1
Number of basis functions: 19
Number of orthogonalized basis functions: 19
Number of auxiliary basis functions: 109
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> GHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 19
Number of auxiliary basis functions: 109
Iter               Energy           ΔE       ||ΔD||  ||AO

GHF(charge=1, do_diis=True, diis_start=1, diis_nvec=8, diis_min=2, econv=1e-09, dconv=1e-06, maxiter=100, guess_type='minao', level_shift=None, level_shift_thresh=1e-05, die_if_not_converged=True, executed=True, converged=True, ms_guess=None, guess_mix=False, alpha_beta_mix=False, break_complex_symmetry=False, j_adapt=False)

In [8]:
# total number of spinorbitals
norb = ghf.nmo * 2

# freeze 1s orbital of fluorine
core_orbitals = [0, 1]

# orbitals are all orbitals except the core orbitals
orbitals = sorted(list(set(range(norb)) - set(core_orbitals)))

ghf_mo_integrals = SpinorbitalIntegrals(system=system, C=ghf.C[0],spinorbitals=orbitals, core_spinorbitals=core_orbitals)

In [9]:
# number of occupied orbitals minus the core orbitals
nocc = ghf.nel - len(core_orbitals)

# extract the occupied blocks of the one- and two-electron integrals
Hocc = ghf_mo_integrals.H[:nocc, :nocc]
Vocc = ghf_mo_integrals.V[:nocc, :nocc, :nocc, :nocc] 

# compute the Hartree-Fock energy using the integrals
Eghf = ghf_mo_integrals.E + np.einsum("ii->",Hocc) + 0.5 * np.einsum("ijij->", Vocc) - 0.5 *np.einsum("ijji->", Vocc)

print(f"GHF energy from SCF:       {ghf.E:.12f} Eh")
print(f"GHF energy from integrals: {Eghf.real:.12f} Eh")

GHF energy from SCF:       -99.503004679661 Eh
GHF energy from integrals: -99.503004679661 Eh


## Integrals and density matrices from complete-active-space self-consistent-field (CASSCF) computations

Next we consider how to extract integrals and density matrices from CASSCF computations. Let's start by setting up a computation on the HF molecule:

In [10]:
from forte2 import MCOptimizer, State

xyz = """
N 0.0 0.0 0.0
N 0.0 0.0 1.4
"""

system = System(xyz=xyz, basis_set="cc-pVDZ", auxiliary_basis_set="cc-pVTZ-JKFIT")
guess_rhf = RHF(charge=0, econv=1e-12)(system)
casscf = MCOptimizer(
    State(nel=14, multiplicity=1, ms=0.0),
    active_orbitals=[4, 5, 6, 7, 8, 9],
    core_orbitals=[0, 1, 2, 3],
    gconv=1e-7,
)(guess_rhf)
casscf.run()


Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   N   0.00000000   0.00000000   0.00000000
   N   0.00000000   0.00000000   2.64561657
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 2.861e+00
  Min eigenvalue: 1.728e-02
  Condition number: 1.655e+02
  Inverse condition number: 6.041e-03
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 1.728e-02
Number of electrons: 14
Number of alpha electrons: 7
Number of beta electrons: 7
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: 158
Energy convergence criterion: 1.000000e-12
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 28
Number of auxiliary basis function

MCOptimizer(states=State(multiplicity=1, ms=0.0, nel=14, system=None, charge=0, gas_min=[], gas_max=[], symmetry=0, symmetry_label=None, na=7, nb=7, twice_ms=0), nroots=1, weights=[[1.0]], mo_space=MOSpace(nmo=np.int64(28), active_orbitals=[[4, 5, 6, 7, 8, 9]], core_orbitals=[0, 1, 2, 3], frozen_core_orbitals=[], frozen_virtual_orbitals=[]), frozen_core_orbitals=None, core_orbitals=[0, 1, 2, 3], active_orbitals=[4, 5, 6, 7, 8, 9], frozen_virtual_orbitals=None, final_orbital='semicanonical', ci_algorithm='hz', die_if_not_converged=True, active_frozen_orbitals=None, freeze_inter_gas_rots=False, maxiter=50, econv=1e-08, gconv=1e-07, micro_maxiter=6, max_rotation=0.2, ci_maxiter=100, ci_econv=1e-12, ci_rconv=1e-06, ci_guess_per_root=2, ci_ndets_per_guess=10, ci_collapse_per_root=2, ci_basis_per_root=4, ci_energy_shift=None, do_transition_dipole=False, converged=True, executed=True, two_component=False)

From the `MCOptimizer` object we can extract the integrals in the molecular orbital basis:

In [11]:
casscf_mo_integrals = RestrictedMOIntegrals(system=system, C=casscf.C[0], orbitals=[4, 5, 6, 7, 8, 9], core_orbitals=[0, 1, 2, 3])

### Spin-dependent reduced density matrices

The `MCOptimizer` class also exposes reduced density matrices. These quantities are computed only for the **active orbitals** specified when calling `MCOptimizer` and therefore ignore contributions from the doubly occupied (core) orbitals.

We begin by considering spin-dependent quantities. The following quantities are defined over **spatial orbital indices**:
- **One-body reduced density matrix (1-RDM)**. There are two spin-dependent contributions:
\begin{align}
\gamma^{\alpha}_{pq} = \langle \Psi | \hat{a}_{p\alpha}^{\dagger} \hat{a}_{q\alpha} | \Psi \rangle, \\
\gamma^{\beta}_{pq} = \langle \Psi | \hat{a}_{p\beta}^{\dagger} \hat{a}_{q\beta} | \Psi \rangle.
\end{align}

- **Two-body reduced density matrix (2-RDM)**. There are three unique spin-dependent contributions:
\begin{align}
\gamma^{\alpha\alpha}_{pqrs} = \langle \Psi | \hat{a}_{p\alpha}^{\dagger} \hat{a}_{q\alpha}^{\dagger} \hat{a}_{s\alpha} \hat{a}_{r\alpha} | \Psi \rangle, \\
\gamma^{\alpha\beta}_{pqrs} = \langle \Psi | \hat{a}_{p\alpha}^{\dagger} \hat{a}_{q\beta}^{\dagger} \hat{a}_{s\beta} \hat{a}_{r\alpha} | \Psi \rangle, \\
\gamma^{\beta\beta}_{pqrs} = \langle \Psi | \hat{a}_{p\beta}^{\dagger} \hat{a}_{q\beta}^{\dagger} \hat{a}_{s\beta} \hat{a}_{r\beta} | \Psi \rangle.
\end{align}

**Important**: To save space, Forte2 stores only the nonredundant elements of the αα and ββ 2-RDMS as a matrix with compound indices $[pq]$ and $[rs]$ where $p>q$ and $r>s$: 
\begin{align}
\gamma^{\alpha\alpha}_{[pq],[rs]} = \gamma^{\alpha\alpha}_{pq,rs}
\end{align}

The mapping between the compound index $[pq]$ and the spatial orbital indices $p$ and $q$ is given by (assuming $p>q$):
\begin{equation}
[pq] = \frac{p(p-1)}{2} + q
\end{equation}
or equivalently:
| $[pq]$ | $p$ | $q$ |
| ------ | --- | --- |
| 0 | 1 | 0 |
| 1 | 2 | 0 |
| 2 | 2 | 1 |
| 3 | 3 | 0 |
| ⋮ | ⋮ | ⋮ |

To obtain the full 2-RDM in the αα and ββ spin sectors, we can use the helper function `cpp_helpers.packed_tensor4_to_tensor4` which takes as input the nonredundant 2-RDM and returns a 4-index tensor with dimensions defined over spatial orbital indices.
This operation can be expensive for large active spaces since the size of the redundant 2-RDM grows as $N_\mathrm{A}^4$ where $N_\mathrm{A}$ is the number of active orbitals.

The following code shows how to obtain the spin-dependent 1- and 2-RDMs and prints the shape of each quantity:

In [13]:
from forte2 import cpp_helpers

# the spin-dependent 1-RDMs stored as 2D arrays with spatial orbital indices
g1a, g1b = casscf.make_sd_1rdm(0)
print(f"Shape of g1a: {g1a.shape}")
print(f"Shape of g1b: {g1b.shape}")

# the spin-dependent 2-RDMs stored as 2D, 4D, and 2D arrays with spatial orbital indices
g2aa, g2ab, g2bb = casscf.make_sd_2rdm(0)
print(f"Shape of g2ab: {g2ab.shape}")
print(f"Shape of g2aa: {g2aa.shape}")
print(f"Shape of g2bb: {g2bb.shape}")

g2aa = cpp_helpers.packed_tensor4_to_tensor4(g2aa)
g2bb = cpp_helpers.packed_tensor4_to_tensor4(g2bb)   

print(f"Shape of g2aa (unpacked): {g2aa.shape}")
print(f"Shape of g2bb (unpacked): {g2bb.shape}")

Shape of g1a: (6, 6)
Shape of g1b: (6, 6)
Shape of g2ab: (6, 6, 6, 6)
Shape of g2aa: (15, 15)
Shape of g2bb: (15, 15)
Shape of g2aa (unpacked): (6, 6, 6, 6)
Shape of g2bb (unpacked): (6, 6, 6, 6)


Note that the function `make_sd_1rdm` can be used for both state-specific and state-averaged computations. Its signature is:
```python
make_sd_1rdm(left_root: int, right_root: int | None = None, solver_index: int | None = None) -> Tuple[np.ndarray, np.ndarray]
```


- In the **state-specific case**, only the `left_root` parameter needs to be specified and the 1-RDM will be computed as $\langle \Psi_\mathrm{left} | \hat{a}_{p\sigma}^{\dagger} \hat{a}_{q\sigma} | \Psi_\mathrm{left} \rangle$.

- In the **state-averaged case**, both `left_root` and `right_root` need to be specified and the 1-RDM will be computed as $\langle \Psi_\mathrm{left} | \hat{a}_{p\sigma}^{\dagger} \hat{a}_{q\sigma} | \Psi_\mathrm{right} \rangle$.
The `solver_index` parameter specifies the index of the CI solver to use, if we are using multiple CI solvers to average over different spin and GAS restrictions. If only one CI solver is used, this parameter can be left as `None` and the default solver will be used

From these quantities we can verify that the CASSCF energy is consistent with the expression:

\begin{equation}
\begin{split}
E_\mathrm{CASSCF} = & \ E_\mathrm{core} + \sum_{i}^{\mathbf{A}} h_{ij} \gamma^{\alpha}_{ij} +
\sum_{i}^{\mathbf{A}} h_{ij} \gamma^{\beta}_{ij} \\
& + \frac{1}{2} \sum_{ijkl}^{\mathbf{A}} \langle ij | kl\rangle \gamma^{\alpha\alpha}_{ijkl}
+ \frac{1}{2} \sum_{ijkl}^{\mathbf{A}} \langle ij | kl\rangle \gamma^{\beta\beta}_{ijkl}
+ \sum_{ijkl}^{\mathbf{A}} \langle ij | kl\rangle \gamma^{\alpha\beta}_{ijkl}.
\end{split}
\end{equation}

In [14]:
Ecasscf = casscf_mo_integrals.E
Ecasscf += np.einsum('ij,ji->', g1a, casscf_mo_integrals.H)
Ecasscf += np.einsum('ij,ji->', g1b, casscf_mo_integrals.H)
Ecasscf += 0.5 * np.einsum('ijkl,ijkl->', g2aa, casscf_mo_integrals.V)
Ecasscf += 0.5 * np.einsum('ijkl,ijkl->', g2bb, casscf_mo_integrals.V)
Ecasscf += np.einsum('ijkl,ijkl->', g2ab, casscf_mo_integrals.V)

print(f"CASSCF energy from CASSCF:    {casscf.E:.12f} Eh")
print(f"CASSCF energy from integrals: {Ecasscf:.12f} Eh")

CASSCF energy from CASSCF:    -108.980048415574 Eh
CASSCF energy from integrals: -108.980048415583 Eh


### Spin-free reduced density matrices

We can similarly access spin-free reduced density matrices defined over **spatial orbital indices**:
- **One-body spin-free reduced density matrix (1-RDM)**. There are two spin-dependent contributions:
\begin{equation}
\Gamma_{pq} = \gamma^{\alpha}_{pq} + \gamma^{\beta}_{pq}
\end{equation}

- **Two-body spin-free reduced density matrix (2-RDM)**. There are three unique spin-dependent contributions:
\begin{align}
\Gamma_{pqrs} = \gamma^{\alpha\alpha}_{pqrs} + \gamma^{\alpha\beta}_{pqrs} + \gamma^{\alpha\beta}_{qpsr} + \gamma^{\beta\beta}_{pqrs}
\end{align}

These quantities can be obtained via the methods `make_sf_1rdm` and `make_sf_2rdm` of the `MCOptimizer` class. Below we compute these quantities and print their shapes for the first state of the CASSCF computation:

In [16]:
# the spin-dependent 1-RDMs stored as 2D arrays with spatial orbital indices
G1 = casscf.make_sf_1rdm(0)
print(f"Shape of G1: {G1.shape}")

# the spin-dependent 2-RDMs stored as 2D, 4D, and 2D arrays with spatial orbital indices
G2 = casscf.make_sf_2rdm(0)
print(f"Shape of G2: {G2.shape}")

Shape of G1: (6, 6)
Shape of G2: (6, 6, 6, 6)


We can then verify that the CASSCF energy is consistent with the spin-free expression:

\begin{equation}
E_\mathrm{CASSCF} = E_\mathrm{core} + \sum_{i}^{\mathbf{A}} h_{ij} \Gamma_{ij}  + \frac{1}{2} \sum_{ijkl}^{\mathbf{A}} \langle ij | kl\rangle \Gamma_{ijkl}.
\end{equation}

In [17]:
Ecasscf_sf = casscf_mo_integrals.E
Ecasscf_sf += np.einsum('ij,ji->', casscf_mo_integrals.H, G1)
Ecasscf_sf += 0.5 * np.einsum('ijkl,ijkl->', casscf_mo_integrals.V, G2)

print(f"CASSCF energy from CASSCF:    {casscf.E:.12f} Eh")
print(f"CASSCF energy from integrals: {Ecasscf_sf:.12f} Eh")

CASSCF energy from CASSCF:    -108.980048415574 Eh
CASSCF energy from integrals: -108.980048415583 Eh
